In [ ]:
from pathlib import Path
from ebooklib import epub
import re

# Folder containing TXT files
INPUT_DIR = Path(".")

# Match filenames like:
# EasyItalianNews_2026-06-02.txt
pattern = re.compile(r"EasyItalianNews_(\d{4}-\d{2}-\d{2})\.txt")

# Collect and sort files by date
files = []

for file in INPUT_DIR.glob("EasyItalianNews_*.txt"):
    match = pattern.match(file.name)
    if match:
        files.append((match.group(1), file))

files.sort(key=lambda x: x[0])

In [ ]:
files

In [ ]:
# Create EPUB book
book = epub.EpubBook()

book.set_identifier("easy-italian-news")
book.set_title("Easy Italian News 2026 Collection")
book.set_language("it")
book.add_author("Easy Italian News")

chapters = []

for idx, (date_str, file_path) in enumerate(files, start=1):
    text = file_path.read_text(encoding="utf-8")

    # Convert plain text to simple HTML
    html_content = "<br/>".join(
        line.strip() for line in text.splitlines()
    )

    chapter = epub.EpubHtml(
        title=date_str,
        file_name=f"chapter_{idx}.xhtml",
        lang="it"
    )

    chapter.content = f"""
    <h1>{date_str}</h1>
    <p>{html_content}</p>
    """

    book.add_item(chapter)
    chapters.append(chapter)

# Table of contents
book.toc = chapters

# Navigation files
book.add_item(epub.EpubNcx())
book.add_item(epub.EpubNav())

# Spine
book.spine = ["nav"] + chapters

# Save EPUB
epub.write_epub("EasyItalianNews.epub", book)

print(f"Created EPUB with {len(chapters)} chapters")
